# 2.9 Object Identity, Aliasing, and Mutability

## Advanced / Expert-Level Python — Quantitative Research Engineering

### Overview

Python programs manipulate **objects**, while names, attributes, and containers provide references to those objects. Two expressions may compare equal without identifying the same object; distinct containers may share nested mutable values; and copying may deliberately preserve or break parts of an object graph. Correctness depends on understanding which references are shared, which operations mutate existing objects, and which operations construct independent state.

This module examines identity, equality, hashability, mutability, shallow and deep copying, cycles, custom copy protocols, function argument sharing, weak references, read-only views, structural sharing, and concurrency. It then develops a **point-in-time market-data revision and portfolio valuation system** that performs immutable, auditable, copy-on-write data revisions, detects source-data contamination, and verifies financial invariants.

### Learning Objectives

1. Distinguish object identity (`is`, `id`) from equality (`==`) and explain why equality may invoke arbitrary user-defined methods.
2. Identify aliases across nested object graphs, including repeated references and cycles.
3. Distinguish immutable *containers*, immutable *contents*, and hashable values.
4. Compare assignment, shallow copying, deep copying, and immutable structural sharing.
5. Explain how `deepcopy` memoization preserves alias relationships and handles cycles.
6. Implement custom `__copy__` and `__deepcopy__` without violating ownership contracts.
7. Diagnose mutation-before-failure, augmented assignment, argument sharing, and mutable defaults.
8. Explain the `__eq__`/`__hash__` contract and dictionary-key hazards.
9. Use `weakref`, `gc`, and allocation diagnostics without assuming implementation-specific finalization timing.
10. Evaluate `MappingProxyType`, frozen dataclasses, `dataclasses.replace`, and copy-on-write designs.
11. Test logically shared data under deterministic concurrent modifications.
12. Build immutable, content-addressed financial market snapshots with checked revisions and pure valuation.
13. Develop identity, mutation, isolation, accounting, and rollback invariants.
14. State precisely what Python guarantees and what is only a CPython implementation detail.

**Environment:** Python 3.10+; standard library only. **Difficulty:** Advanced / Expert.  
**Research standard:** Examples are synthetic and educational, not live investment advice.  
**Scope boundary:** Detailed assignment operator protocol is in §2.5; binding and scope rules are in §§2.1–2.8. Here the emphasis is **shared object graphs and state integrity**.

### Recommended Reading

- Python Language Reference — Data Model: https://docs.python.org/3/reference/datamodel.html
- Python `copy` module: https://docs.python.org/3/library/copy.html
- Python `dataclasses` module: https://docs.python.org/3/library/dataclasses.html
- Python `weakref` module: https://docs.python.org/3/library/weakref.html
- Python `gc` and `tracemalloc`: https://docs.python.org/3/library/gc.html and https://docs.python.org/3/library/tracemalloc.html

The notebook is self-contained; external links are optional reference reading.

## Notebook Roadmap

| Section | Advanced focus |
|---|---|
| 2.9.1 | Identity, equality, and runtime reference semantics |
| 2.9.2 | Aliasing graphs and nested mutable structures |
| 2.9.3 | Mutability, deep immutability, and hashability |
| 2.9.4 | Shallow copies and view semantics |
| 2.9.5 | Deep copying, memoization, and cycles |
| 2.9.6 | Custom copying and intentional sharing |
| 2.9.7 | Augmented assignment and mutation-before-failure |
| 2.9.8 | Call-by-sharing, defaults, and closure state |
| 2.9.9 | Equality and hashing under mutation |
| 2.9.10 | Weak references and object lifetime |
| 2.9.11 | Frozen records, structural sharing, and copy-on-write |
| 2.9.12 | Read-only facades vs. deeply immutable state |
| 2.9.13 | Reference introspection and memory diagnostics |
| 2.9.14 | Concurrent read–modify–write and alias safety |
| 2.9.15 | Practical Application — Point-in-Time Market Data Revision Engine |
| 2.9.16 | Automated Testing and Financial Invariants |
| 2.9.17 | Final Analytical Interpretation |
| — | 12 Advanced Practice Exercises + Expert Challenge |

**Notebook contract:** Every foundational code example is followed by its own `### **Analytical Interpretation**` section. Examples include explicit assertions; numerical results are verified by the notebook test suite.

## 2.9.1 Identity, Equality, and Reference Semantics

### Theoretical Foundations


### Formal Model

Every object has **identity**, **type**, and **value**. The `is` operator tests identity. The `==` operator invokes the equality protocol and may return a user-defined result; do not replace it with `is` except for intentional singleton/reference tests (`x is None`, private sentinels). `id(x)` is stable and unique **during the lifetime of x**, but its numeric value is not a persistent key or universal memory address. CPython often uses an address; that is not a Python-language guarantee.

- **Aliasing:** two references target the same live object.
- **Value equality:** two objects satisfy the type's equality semantics.
- **Rebinding:** one name changes its target object without modifying its former target.
- **Identity preservation:** a mutable object can change value while retaining identity.
- **Singleton sentinel:** use a private `object()` instance when `None` is a meaningful value.

Avoid reasoning from small integer caching, interned strings, or optimizer-created constants: an implementation may reuse immutable values in different ways.

In [1]:
from dataclasses import dataclass

left = [100, 200]
right = [100, 200]
alias = left
assert left == right and left is not right
assert alias is left
left.append(300)
assert alias == [100, 200, 300] and right == [100, 200]

MISSING = object()
cache = {'AAPL': None}
assert cache.get('AAPL', MISSING) is None
assert cache.get('MSFT', MISSING) is MISSING

@dataclass(frozen=True)
class PriceLevel:
    ticker: str
    price: int

a = PriceLevel('AAPL', 200)
b = PriceLevel('AAPL', 200)
assert a == b and a is not b
print('identity shares object:', alias is left)
print('value-equal frozen records distinct:', a == b, a is b)
print('missing distinct from present None:', cache.get('MSFT', MISSING) is MISSING)

identity shares object: True
value-equal frozen records distinct: True False
missing distinct from present None: True


### **Analytical Interpretation**


1. **Equality does not imply aliasing.** Two separately allocated lists initially compare equal, yet only one is modified.
2. **Mutation preserves identity.** `append` alters the original list visible through `alias`.
3. **Private sentinels disambiguate absence.** The same dictionary can store a legitimate `None` and separately represent a missing key.
4. **Value objects can be equal but distinct.** Dataclass-generated equality compares fields; `is` still checks references.
5. **Quant implication:** identifying duplicated data by object identity is different from detecting equal market observations. Neither alone establishes correct provenance.

## 2.9.2 Aliasing Graphs, Nested Containers, and Cycles

### Theoretical Foundations


### The Object-Graph Perspective

Model a running program as a directed graph: names, container slots, attributes, and closure cells are **reference edges**; objects are nodes. Several paths can reach the same object. Shallow-looking operations may therefore produce *deep* changes. Cyclic graphs are legal; naïve recursive traversal may never terminate.

For nested financial inputs `portfolio['prices']`, `portfolio` and its nested `prices` list have distinct identities. A second outer dictionary can nonetheless point to that same nested list. Graph diagnostics should track **live objects** and avoid interpreting equality as an identity relation.

**Caveat:** manually exploring `id()` values without maintaining live references can be misleading because IDs can be reused after collection.

In [2]:
shared_prices = [100, 101, 102]
feed_a = {'ticker': 'AAPL', 'prices': shared_prices}
feed_b = {'ticker': 'AAPL', 'prices': shared_prices}
assert feed_a is not feed_b
assert feed_a['prices'] is feed_b['prices']

feed_a['prices'][0] = 99
assert feed_b['prices'][0] == 99

graph = []
graph.append(graph)  # deliberate one-node cycle
assert graph[0] is graph

def contains_cycle_list(node: list) -> bool:
    # Specific to a direct self-reference; not a general graph algorithm.
    return any(item is node for item in node)

assert contains_cycle_list(graph)
print('outer objects distinct:', feed_a is not feed_b)
print('nested prices shared:', feed_a['prices'] is feed_b['prices'])
print('self-cycle detected:', contains_cycle_list(graph))

outer objects distinct: True
nested prices shared: True
self-cycle detected: True


### **Analytical Interpretation**


1. **Outer independence is insufficient:** different dictionaries can still share the same list.
2. **Reachability determines exposure:** a mutation through either feed reaches the same nested list.
3. **Cycles require explicit traversal policy:** simple recursive tree walkers can recurse forever on general cyclic graphs.
4. **Risk to ML pipelines:** a normalization routine applied through one view may contaminate another experiment if both reference the same underlying data.
5. **Testing strategy:** assert identity relationships in addition to numerical results; avoid assuming `copy()` isolates all nested objects.

## 2.9.3 Mutability, Deep Immutability, and Hashability

### Theoretical Foundations


### Separate Concepts

**Mutability** asks whether the observable value/state can change through supported operations. **Hashability** asks whether `hash(x)` can be computed consistently for use in hashed collections. They are not interchangeable: immutable tuples containing unhashable lists are themselves unhashable; user-defined mutable objects may be identity-hashable, but using mutable value-based hashes as mapping keys violates the stable-hash contract. An immutable outer container does **not** ensure its reachable graph is immutable.

- `tuple` fixes its references to elements, not the values of mutable referenced elements.
- `frozenset` is immutable, but elements must themselves be hashable.
- `@dataclass(frozen=True)` prevents ordinary attribute reassignment; its fields can still refer to mutable objects.
- Hash equality invariant: `a == b` implies `hash(a) == hash(b)` when both are hashable.

For portfolio identifiers, use stable immutable keys; store changing market prices as values instead of mutating keys.

In [3]:
from dataclasses import dataclass

outer = ('AAPL', {'close': [100, 101]})
outer[1]['close'].append(102)
assert outer[1]['close'] == [100, 101, 102]

try:
    hash((1, [2, 3]))
except TypeError:
    print('tuple containing a list is unhashable')
else:
    raise AssertionError('expected TypeError')

@dataclass(frozen=True)
class FrozenButShallow:
    prices: list[int]

record = FrozenButShallow([100, 101])
record.prices.append(102)
assert record.prices[-1] == 102
try:
    hash(record)
except TypeError:
    print('generated hash encounters unhashable nested list')
else:
    raise AssertionError('expected TypeError')

assert hash(('AAPL', 100)) == hash(('AAPL', 100))
print('frozen record nested list:', record.prices)

tuple containing a list is unhashable
generated hash encounters unhashable nested list
frozen record nested list: [100, 101, 102]


### **Analytical Interpretation**


1. **Container immutability is shallow.** The tuple cannot replace its dict element, but the nested dict/list can change.
2. **Hashing depends on elements.** A tuple is hashable only if each element supports hashing; `frozen=True` does not make unhashable fields hashable.
3. **The generated dataclass hash is not a deep freeze.** A hash attempt may raise `TypeError` when a field contains a list.
4. **Financial implications:** put immutable ticker or timestamp identifiers in dictionary keys and maintain mutable observations separately.
5. **Model contract:** deep immutability requires deliberately designing the full reachable data graph, not setting a single decorator flag.

## 2.9.4 Shallow Copy, Views, and Partial Independence

### Theoretical Foundations


### Copy Semantics

Name assignment shares a reference. `copy.copy()` and methods such as `dict.copy()` usually create a new outer container and reuse references to its contents. A **read-only mapping proxy** is different: it is a live view of the original mapping, not a snapshot. If the original dictionary changes, the proxy shows the update.

For NumPy/Pandas (later modules), **views** and **copy-on-write semantics** add further complexity, but no such dependency is required here.

Memory/time: shallow container copies are typically proportional to the number of outer elements, not the full recursively referenced graph; implementation and custom types may differ.

In [4]:
import copy
from types import MappingProxyType

source = {'AAPL': {'prices': [100, 101]}}
shallow = copy.copy(source)
proxy = MappingProxyType(source)
assert shallow is not source
assert shallow['AAPL'] is source['AAPL']

source['AAPL']['prices'].append(102)
assert shallow['AAPL']['prices'][-1] == 102
assert proxy['AAPL']['prices'][-1] == 102

source['MSFT'] = {'prices': [200]}
assert 'MSFT' not in shallow and 'MSFT' in proxy
try:
    proxy['JPM'] = {'prices': [150]}
except TypeError:
    print('proxy rejects assignment through its interface')
else:
    raise AssertionError('expected TypeError')
print('outer shallow keys:', sorted(shallow))
print('live proxy keys:', sorted(proxy))

proxy rejects assignment through its interface
outer shallow keys: ['AAPL']
live proxy keys: ['AAPL', 'MSFT']


### **Analytical Interpretation**


1. **Shallow copying creates only outer independence.** Existing nested objects remain shared.
2. **A proxy is not a frozen snapshot.** It disallows updates *through the proxy*, yet reflects changes made to its backing dictionary.
3. **New keys distinguish the semantics.** Added `MSFT` is visible in the live proxy but absent from the previously copied dictionary.
4. **Quant implication:** `MappingProxyType(external_dict)` cannot guarantee reproducibility if an upstream loader keeps modifying that dict.
5. **Safe design:** construct a private owned mapping and deeply immutable values before exposing a read-only view.

## 2.9.5 Deep Copy, Memoization, and Cyclic Graphs

### Theoretical Foundations


### `copy.deepcopy` Graph Semantics

A deep copy recursively reconstructs supported objects, but it is **not** simply equivalent to recursively allocating a completely distinct object for every reference. The memo dictionary preserves internal aliasing of repeated references and prevents infinite recursion through cycles. Certain objects (modules, functions, etc.) are deliberately not cloned in the ordinary way. Deep copy may be expensive or duplicate resources that should stay shared.

Define the desired contract: **clone mutable owned state, preserve intentionally shared immutable configuration or resources**. Test both external isolation and internal reference topology.

In [5]:
import copy

shared = {'AAPL': [100, 101]}
source = [shared, shared]
source.append(source)  # cycle plus shared nested object
cloned = copy.deepcopy(source)

assert cloned is not source
assert cloned[0] is cloned[1]  # memo preserves alias topology
assert cloned[0] is not shared
assert cloned[2] is cloned  # cycle reconstructed

cloned[0]['AAPL'].append(102)
assert shared['AAPL'] == [100, 101]
assert cloned[1]['AAPL'] == [100, 101, 102]
print('shared inside clone:', cloned[0] is cloned[1])
print('clone isolated from original:', cloned[0] is not source[0])
print('self-cycle reconstructed:', cloned[2] is cloned)

shared inside clone: True
clone isolated from original: True
self-cycle reconstructed: True


### **Analytical Interpretation**


1. **Memo preserves repeated edges.** Both locations pointing to one source dictionary still point to one cloned dictionary.
2. **External isolation and internal sharing coexist.** The clone does not share the nested mutable dict with the source, but still shares it internally.
3. **Cycles are reconstructed.** The cloned list points to itself rather than to the source list.
4. **A deep copy is not automatically a valid financial snapshot.** Source objects such as open handles, callbacks, and externally owned caches may require special policy.
5. **Performance:** full deep copies can multiply allocations for large price matrices; immutable structural sharing can be more efficient.

## 2.9.6 Custom `__copy__` / `__deepcopy__` and Ownership Contracts

### Theoretical Foundations


### User-Defined Copy Behavior

`copy.copy(obj)` may invoke `obj.__copy__()`. `copy.deepcopy(obj)` may invoke `obj.__deepcopy__(memo)`. Correct implementations must respect the `memo` table: register the new object before recursively copying references when cycles may occur; pass the same memo to nested `deepcopy`. Immutable configuration objects may intentionally be reused.

**Anti-pattern:** return `self` from `__deepcopy__` for an object containing mutable owned state unless that sharing is explicitly guaranteed safe.

In [6]:
import copy
from dataclasses import dataclass

@dataclass(frozen=True, slots=True)
class ImmutablePolicy:
    max_weight: float

class ResearchBundle:
    def __init__(self, prices, policy):
        self.prices = prices
        self.policy = policy
    def __copy__(self):
        return type(self)(self.prices, self.policy)
    def __deepcopy__(self, memo):
        existing = memo.get(id(self))
        if existing is not None:
            return existing
        result = object.__new__(type(self))
        memo[id(self)] = result
        result.prices = copy.deepcopy(self.prices, memo)
        result.policy = self.policy  # deliberate immutable sharing
        return result

bundle = ResearchBundle({'AAPL': [100, 101]}, ImmutablePolicy(0.25))
shallow_bundle = copy.copy(bundle)
deep_bundle = copy.deepcopy(bundle)
assert shallow_bundle.prices is bundle.prices
assert deep_bundle.prices is not bundle.prices
assert deep_bundle.policy is bundle.policy
bundle.prices['AAPL'].append(102)
assert deep_bundle.prices['AAPL'] == [100, 101]
print('shared immutable policy:', deep_bundle.policy is bundle.policy)
print('prices isolated:', deep_bundle.prices is not bundle.prices)

shared immutable policy: True
prices isolated: True


### **Analytical Interpretation**


1. **Shallow copy delegates ownership policy to the class.** This example shares mutable prices intentionally, making its limitations visible.
2. **Deep copy registers the new object before recursion.** The memo supports repeated references and cycles.
3. **Intentional sharing is valid for safely immutable values.** The frozen scalar-only policy is reused to avoid redundant duplication.
4. **`frozen` is not a universal security guarantee.** A policy containing nested mutable fields would require stronger protections.
5. **Quant engineering:** copy behavior should be documented for market buffers, caches, policies, and model parameters.

## 2.9.7 Augmented Assignment and Mutation Before Failure

### Theoretical Foundations


### Nonatomic-Looking Expressions

`x += y` may call `x.__iadd__(y)` and mutate `x` before Python assigns the result back to the target. With a mutable list referenced inside an immutable tuple, the in-place list operation may succeed, then the required tuple item assignment fails. **Catching the exception does not restore the old list contents.**

This is a targeted review of the critical §2.5 behavior from the perspective of aliases and object graph integrity.

In [7]:
record = ([100, 200],)
prices_alias = record[0]
try:
    record[0] += [300]
except TypeError:
    print('tuple target rejected assignment-back')
else:
    raise AssertionError('expected TypeError')

assert record[0] is prices_alias
assert record[0] == [100, 200, 300]

safe_candidate = tuple(record[0] + [400])
assert safe_candidate == (100, 200, 300, 400)
assert record[0] == [100, 200, 300]
print('mutation survived failed assignment:', record[0])

tuple target rejected assignment-back
mutation survived failed assignment: [100, 200, 300]


### **Analytical Interpretation**


1. **Two effects occur:** in-place mutation of the list, then failed assignment to an immutable tuple slot.
2. **The exception does not roll back changes.** All aliases to the list see the appended item.
3. **Safer approach:** compute a new immutable result and publish it only when ready.
4. **Financial consequence:** ledger data may be modified even though the operation that attempted to update it raised an exception.
5. **Design requirement:** use staged data transformations with explicit consistency checks for critical updates.

## 2.9.8 Call-by-Sharing, Mutable Defaults, and Closures

### Theoretical Foundations


### Function Calls Do Not Clone Inputs

Python passes references to objects by assigning arguments to local parameter names (sometimes called **call-by-sharing**). Rebinding a parameter does not rebind the caller's name, but mutating a shared object is observable by the caller. Default argument expressions are evaluated at function definition time, not once per call; mutable defaults can accumulate state. Closures can also retain mutable referenced objects.

Professional API design should distinguish *borrowed read-only*, *owned mutable*, and *copied* inputs. Do not treat `typing.Sequence` annotations as runtime immutability enforcement.

In [8]:
def mutate_then_rebind(series: list[int]) -> list[int]:
    series.append(3)
    series = [99]  # only local name is rebound
    return series

prices = [1, 2]
returned = mutate_then_rebind(prices)
assert prices == [1, 2, 3]
assert returned == [99]

def flawed_buffer(value: int, buffer: list[int] = []):
    buffer.append(value)
    return buffer

first = flawed_buffer(10)
second = flawed_buffer(20)
assert first is second and second == [10, 20]

def safe_buffer(value: int, buffer: list[int] | None = None):
    owned = [] if buffer is None else buffer
    owned.append(value)
    return owned

assert safe_buffer(10) is not safe_buffer(20)
print('caller list changed:', prices)
print('shared default accumulated:', second)

caller list changed: [1, 2, 3]
shared default accumulated: [10, 20]


### **Analytical Interpretation**


1. **Local rebinding does not change the caller's binding.** The original list nevertheless records the earlier append.
2. **Default objects persist across invocations.** The flawed implementation creates a long-lived shared list.
3. **Explicit ownership works better.** The safe version creates a fresh list when no container is supplied.
4. **Even the safe version mutates a supplied list.** Its contract should say so; use `list(buffer)` if defensive copying is required.
5. **Research risk:** shared defaults can leak data between training runs, parameter searches, or notebook executions.

## 2.9.9 Equality–Hash Contract and Mutable Dictionary Keys

### Theoretical Foundations


### Hash Stability

Hash-based containers assume a key's hash remains stable while it is present. If two keys compare equal, their hash values must agree. A custom mutable class whose hash depends on mutable fields can corrupt lookup behavior; it may remain physically stored yet be inaccessible using its own updated value. Conversely, a class implementing value equality without a compatible hash is commonly made unhashable.

Do **not** solve this by setting `unsafe_hash=True` indiscriminately. Prefer immutable value objects for asset identifiers, trade IDs, and time keys.

In [9]:
from dataclasses import dataclass

@dataclass(frozen=True)
class InstrumentKey:
    symbol: str
    venue: str

k1 = InstrumentKey('AAPL', 'NASDAQ')
k2 = InstrumentKey('AAPL', 'NASDAQ')
assert k1 == k2 and hash(k1) == hash(k2)
positions = {k1: 100}
assert positions[k2] == 100

@dataclass
class MutableIdentifier:
    symbol: str

try:
    hash(MutableIdentifier('AAPL'))
except TypeError:
    print('value-equal mutable dataclass is unhashable by default')
else:
    raise AssertionError('expected TypeError')

# Hashing a tuple whose elements include a list also fails.
try:
    {('AAPL', [1, 2]): 100}
except TypeError:
    print('unhashable nested key rejected')
else:
    raise AssertionError('expected TypeError')
print('stable immutable lookup:', positions[k2])

value-equal mutable dataclass is unhashable by default
unhashable nested key rejected
stable immutable lookup: 100


### **Analytical Interpretation**


1. **Separate immutable instances can act as equivalent keys.** Equal `InstrumentKey` objects have equal hashes.
2. **Default safeguards matter.** A value-equal, mutable dataclass is unhashable unless a suitable hash is explicitly supplied.
3. **Hashability is transitive over included components for tuple hashing.** Nested unhashable values cause a `TypeError`.
4. **Do not use price or portfolio weight in a mutable key's hash.** Prices change by design; keys must remain stable.
5. **Financial application:** identify a position using immutable `(ticker, venue)` or a validated immutable record.

## 2.9.10 Weak References, Weak Caches, and Reachability

### Theoretical Foundations


### Liveness Without Ownership

A `weakref.ref` does not keep the referent alive. `WeakValueDictionary` and `WeakKeyDictionary` support certain cache policies but do not provide durable storage. A referent may disappear when no strong references remain; the timing depends on implementation and other references. Not every builtin type is weak-referenceable. For `@dataclass(slots=True)`, `weakref_slot=True` is available from Python 3.11, so this notebook uses a normal class for Python 3.10 compatibility.

**Never rely on garbage collection timing for external resource cleanup.** Use context managers and explicit close/commit operations.

In [10]:
import gc
import weakref

class MarketCacheEntry:
    def __init__(self, digest: str):
        self.digest = digest

entry = MarketCacheEntry('v1')
weak = weakref.ref(entry)
cache = weakref.WeakValueDictionary()
cache['v1'] = entry
assert weak() is entry and cache['v1'] is entry

strong_alias = entry
del entry
assert weak() is strong_alias
# Avoid asserting exact finalization time across Python implementations.
del strong_alias
gc.collect()
assert weak() is None or isinstance(weak(), MarketCacheEntry)
print('weak cache key currently present:', 'v1' in cache)

try:
    weakref.ref([])
except TypeError:
    print('ordinary list instances do not support weak refs')
else:
    raise AssertionError('expected TypeError')

weak cache key currently present: False
ordinary list instances do not support weak refs


### **Analytical Interpretation**


1. **Weak caches are opportunistic.** Entries may vanish after their last strong owner is gone.
2. **Deleting one name is not enough if other strong references exist.** `strong_alias` retains the object until it too is removed.
3. **A weak ref is not a correctness mechanism.** Core market data or audit records should not live solely in a weak cache.
4. **Runtime-dependent timing:** do not assume an object must be collected immediately after `del` on all implementations.
5. **For external files and transactions, explicitly manage lifetime.** Reachability and resource cleanup are separate concerns.

## 2.9.11 Frozen Records, Structural Sharing, and Copy-on-Write

### Theoretical Foundations


### Persistent-Style Data Updates

Copy-on-write constructs a new logical state while preserving the old one. If subobjects are immutable, both versions may safely share unchanged records. `dataclasses.replace` can create a new record with a modified field. Python 3.13+ also has `copy.replace` for supported objects, but `dataclasses.replace` is used here to remain compatible with Python 3.10.

**Structural sharing is safe only where shared data is immutable or effectively protected by a reliable ownership contract.** Staged updates must complete before publishing the new state.

In [11]:
from dataclasses import dataclass, replace
from decimal import Decimal

@dataclass(frozen=True, slots=True)
class ImmutableQuote:
    symbol: str
    close: Decimal

old_aapl = ImmutableQuote('AAPL', Decimal('100'))
old_msft = ImmutableQuote('MSFT', Decimal('200'))
old = (old_aapl, old_msft)
new = (replace(old_aapl, close=Decimal('105')), old_msft)

assert old[0].close == Decimal('100')
assert new[0].close == Decimal('105')
assert old[1] is new[1]  # intentional structural sharing
assert old[0] is not new[0]
print('unchanged MSFT record shared:', old[1] is new[1])
print('revised AAPL record distinct:', old[0] is not new[0])

unchanged MSFT record shared: True
revised AAPL record distinct: True


### **Analytical Interpretation**


1. **Old versions remain readable.** Revised AAPL data is represented by a new immutable record.
2. **Unchanged records may be shared.** The MSFT quote is safe to reuse because its fields are immutable.
3. **Copy-on-write avoids a full recursive copy.** Only changed nodes and their enclosing containers need reconstruction.
4. **Hash/content audit:** version identifiers should reflect the data actually used, not merely a user-supplied label.
5. **Application:** historical corrections can be represented as new datasets without rewriting previous backtest inputs.

## 2.9.12 Read-Only Facades vs. Strong Snapshot Isolation

### Theoretical Foundations


### Frozen Interface Is Not Frozen Data

`MappingProxyType` is a dynamic read-only view. A frozen dataclass prevents ordinary field reassignment. Neither automatically freezes the transitive object graph. A robust snapshot should defensively **construct owned immutable values**, use immutable subrecords, and avoid retaining an externally writable reference to its backing mapping. This is an *application-level immutability contract*, not a security sandbox against arbitrary introspection or deliberately bypassed protections.

Avoid storing a live external `dict` inside a proxy and mistaking it for a historical snapshot.

In [12]:
from dataclasses import dataclass, field
from types import MappingProxyType

@dataclass(frozen=True)
class ShallowFrozenState:
    prices: dict[str, list[int]]

original = {'AAPL': [100, 101]}
shallow_frozen = ShallowFrozenState(original)
original['AAPL'].append(102)
assert shallow_frozen.prices['AAPL'][-1] == 102

live_proxy = MappingProxyType(original)
original['MSFT'] = [200]
assert 'MSFT' in live_proxy

owned = {key: tuple(values) for key, values in original.items()}
protected = MappingProxyType(owned)
original['AAPL'].append(103)
assert protected['AAPL'] == (100, 101, 102)
try:
    protected['AAPL'] = (1,)
except TypeError:
    print('protected snapshot mapping rejects writes')
else:
    raise AssertionError('expected TypeError')
print('owned immutable snapshot:', dict(protected))

protected snapshot mapping rejects writes
owned immutable snapshot: {'AAPL': (100, 101, 102), 'MSFT': (200,)}


### **Analytical Interpretation**


1. **Frozen dataclasses are shallow.** Nested dict/list objects remain mutable.
2. **Read-only proxy is a live view.** It follows updates to the original backing dictionary.
3. **Defensive construction matters.** The owned mapping stores tuples independently reconstructed from input lists.
4. **This example keeps `owned` accessible to local code.** A real snapshot builder must not expose such a mutable backing dictionary.
5. **Quant use:** historical signal evaluation should receive revisioned, immutable data versions rather than a changing provider feed.

## 2.9.13 Reference Introspection and Memory Diagnostics

### Theoretical Foundations


### Tools and Caveats

`sys.getrefcount` is CPython-oriented, counts temporary references, and is absent or different in other runtimes. `gc.get_referrers` exposes selected tracked referents, may show incomplete relationships, and should not be used as a security or correctness oracle. `tracemalloc` tracks Python-managed memory allocations after tracing begins, not arbitrary native array buffers or complete process RSS.

For portable tests, assert **semantic identity** and **value isolation**. Treat memory profiles as measurements dependent on environment, not fixed reference values.

In [13]:
import gc
import sys
import tracemalloc

tracemalloc.start()
tracemalloc.reset_peak()
measurement_start, _ = tracemalloc.get_traced_memory()
probe = [bytearray(128) for _ in range(50)]
measurement_end, peak = tracemalloc.get_traced_memory()
assert measurement_end >= 0 and peak >= measurement_end
print('tracemalloc active:', tracemalloc.is_tracing())
print('observed traced allocation change (bytes):', measurement_end - measurement_start)

obj = []
alias = obj
assert obj is alias
if sys.implementation.name == 'cpython':
    count = sys.getrefcount(obj)
    assert count >= 2
    print('CPython snapshot refcount (nonportable):', count)
print('gc module available:', hasattr(gc, 'collect'))
del probe
tracemalloc.stop()

tracemalloc active: True
observed traced allocation change (bytes): 10049
CPython snapshot refcount (nonportable): 3
gc module available: True


### **Analytical Interpretation**


1. **Object-identity tests are portable.** `obj is alias` does not depend on the numeric `id()` representation.
2. **Reference counts are implementation-specific diagnostics.** Never hardcode an expected count across environments.
3. **Memory metrics are contextual.** Allocators, caches, tracing overhead, and native libraries affect observed memory usage.
4. **A small allocation sample is not a benchmark.** Production profiling requires repeated measurements and realistic workload sizes.
5. **Quant pipelines:** track numerical-buffer ownership separately from memory usage, especially when introducing NumPy and Pandas later.

## 2.9.14 Concurrent Read–Modify–Write and Shared State

### Theoretical Foundations


### Object Aliasing Under Concurrency

Shared mutable state can suffer **lost updates** when two workers separately read, compute, and write a value. Do not assume the Global Interpreter Lock makes an arbitrary read–modify–write sequence atomic. This section uses a barrier to force a deterministic lost-update schedule, then demonstrates lock-guarded updates.

Concurrency and aliasing are distinct: locks protect critical sections only when **all** participating code honors the same synchronization contract. Multi-process or distributed updates require stronger coordination.

In [14]:
from concurrent.futures import ThreadPoolExecutor
from threading import Barrier, Lock

state = {'shares': 0}
barrier = Barrier(2)

def intentionally_unsafe_increment():
    observed = state['shares']
    barrier.wait()  # force both workers to read the same value
    state['shares'] = observed + 1

with ThreadPoolExecutor(max_workers=2) as executor:
    tasks = [executor.submit(intentionally_unsafe_increment) for _ in range(2)]
    for task in tasks:
        task.result()
assert state['shares'] == 1  # one of the two updates was lost

state = {'shares': 0}
lock = Lock()
def safe_increment():
    with lock:
        state['shares'] = state['shares'] + 1
with ThreadPoolExecutor(max_workers=2) as executor:
    tasks = [executor.submit(safe_increment) for _ in range(2)]
    for task in tasks:
        task.result()
assert state['shares'] == 2
print('unsafe deterministic final count: 1')
print('locked final count:', state['shares'])

unsafe deterministic final count: 1
locked final count: 2


### **Analytical Interpretation**


1. **Read–modify–write is multiple logical operations.** Both workers read zero before either writes; their writes overwrite each other.
2. **The barrier makes the failure deterministic.** It is a demonstration of interleaving, not a claim that all unsynchronized operations always fail.
3. **Locks guard the complete transition.** The protected operations correctly produce a count of two.
4. **Identity alone provides no coordination.** Both workers sharing one dictionary does not make their updates consistent.
5. **Financial implication:** duplicate holdings adjustments, cash transfers, or order fills require explicit synchronization or transaction processing.

## 2.9.15 Practical Application — Point-in-Time Market Data Revision and Portfolio Valuation

### Project Objective

Build an **auditable, immutable, content-addressed research dataset**. Financial data revisions must not mutate previously published snapshots or retain aliases to externally mutable feed structures. Changes must be **checked against an expected original value**, staged, then published as new snapshots; unchanged immutable asset histories may be shared safely.

### Architecture

    External mutable feed (dict / list / dict)
                      |
                 Validate input
                      |
        Reconstruct immutable Observation records
                      |
          Canonical MarketSnapshot (v1, digest)
                      |
              /-----------------\\
      Portfolio valuation   Checked Correction Batch
              |                    |
       Baseline NAV              Validate entire batch
                                   |
                            Copy-on-write revision
                                   |
                    MarketSnapshot (v2, parent_digest)
                                   |
                     Revised NAV / impact attribution

### Research Contracts

1. Input prices are **decimal strings or `Decimal`**, never silently converted from binary floats; volumes are nonnegative integers excluding booleans.
2. Asset histories are sorted, have no duplicate sessions, contain at least two observations, and terminate at the chosen `asof` session.
3. Every observation is **point-in-time available** (`session <= asof`). This example assumes a completed closing session; vendor release timestamps and post-close publication delays are outside scope.
4. An immutable snapshot holds a tuple of immutable observations and a deterministic SHA-256 content digest.
5. Corrections require an `expected_old_close` and `expected_parent_digest` to prevent silently overwriting stale state.
6. A batch is *all-or-nothing as a pure function*: errors return no new snapshot and cannot alter the previous one.
7. Revisions preserve identity of **unchanged immutable history tuples** (structural sharing).
8. Revaluation preserves position/cash inputs and reports the specific impact of revised prices.
9. A zero-change correction is a no-op and does not create a misleading extra version.
10. No external file, network, database, or live trading API is modified.

### Financial Definitions

For holdings with shares `q_i` and close `p_i` at the as-of session:

\[
\operatorname{MarketValue}_i=q_i p_i
\]

\[
\operatorname{NAV}=\operatorname{Cash}+\sum_i q_i p_i
\]

For an equal-weight basket over the same observed interval:

\[
R_{\mathrm{EW}}=\frac{1}{N}\sum_i\left(\frac{P_{i,T}}{P_{i,0}}-1\right)
\]

A single-price correction has a deterministic first-order NAV effect `shares × price_delta` under fixed holdings. These are synthetic mark-to-market calculations, not predictive returns or execution P&L.

In [15]:
# ============================================================
# DOMAIN MODELS: IMMUTABLE OBSERVATIONS AND MARKET SNAPSHOTS
# ============================================================

from dataclasses import dataclass, field, replace
from datetime import date
from decimal import Decimal, InvalidOperation
import hashlib
import json


def require_decimal(value: object, *, positive: bool = True) -> Decimal:
    """Return a finite exact Decimal; refuse binary floats and booleans."""
    if isinstance(value, bool) or not isinstance(value, (str, Decimal)):
        raise TypeError('Price must be a decimal string or Decimal.')
    try:
        number = Decimal(value)
    except (InvalidOperation, ValueError) as error:
        raise ValueError('Invalid decimal price.') from error
    if not number.is_finite() or (positive and number <= 0):
        raise ValueError('Price must be finite and positive.')
    return number


@dataclass(frozen=True, slots=True)
class MarketObservation:
    ticker: str
    session: date
    close_usd: Decimal
    volume: int

    def __post_init__(self) -> None:
        if not isinstance(self.ticker, str) or not self.ticker.strip():
            raise ValueError('Invalid ticker.')
        if type(self.session) is not date:
            raise TypeError('Session must be a date.')
        if type(self.close_usd) is not Decimal or not self.close_usd.is_finite() or self.close_usd <= 0:
            raise ValueError('Close must be a finite positive Decimal.')
        if type(self.volume) is not int or self.volume < 0:
            raise ValueError('Volume must be a nonnegative integer.')


@dataclass(frozen=True, slots=True)
class MarketSnapshot:
    asof: date
    version: int
    observations: tuple[tuple[str, tuple[MarketObservation, ...]], ...]
    parent_digest: str | None = None
    digest: str = field(init=False)

    def __post_init__(self) -> None:
        if type(self.asof) is not date:
            raise TypeError('asof must be a date.')
        if type(self.version) is not int or self.version < 1:
            raise ValueError('version must be a positive integer.')
        if self.parent_digest is not None and (
            not isinstance(self.parent_digest, str) or len(self.parent_digest) != 64
        ):
            raise ValueError('parent_digest must be a SHA-256 hex digest or None.')
        if type(self.observations) is not tuple or not self.observations:
            raise ValueError('observations must be a nonempty tuple.')
        tickers = []
        canonical = []
        for pair in self.observations:
            if type(pair) is not tuple or len(pair) != 2:
                raise TypeError('Each instrument series must be a pair.')
            ticker, series = pair
            if not isinstance(ticker, str) or not ticker.strip():
                raise ValueError('Invalid instrument ticker.')
            if type(series) is not tuple or len(series) < 2:
                raise ValueError('Each instrument must contain at least two observations.')
            if not all(type(x) is MarketObservation and x.ticker == ticker for x in series):
                raise ValueError('Invalid market observation or ticker mismatch.')
            sessions = [x.session for x in series]
            if sessions != sorted(set(sessions)):
                raise ValueError('Sessions must increase strictly without duplicates.')
            if series[-1].session != self.asof:
                raise ValueError('Each series must end at the as-of session.')
            tickers.append(ticker)
            canonical.append([
                ticker,
                [[x.session.isoformat(), str(x.close_usd), x.volume] for x in series],
            ])
        if tickers != sorted(set(tickers)):
            raise ValueError('Tickers must be sorted and unique.')
        payload = {
            'asof': self.asof.isoformat(),
            'version': self.version,
            'parent_digest': self.parent_digest,
            'observations': canonical,
        }
        wire = json.dumps(payload, sort_keys=True, separators=(',', ':'), ensure_ascii=True)
        object.__setattr__(self, 'digest', hashlib.sha256(wire.encode('utf-8')).hexdigest())


def dataset_rows() -> dict[str, list[dict[str, object]]]:
    """Create new synthetic, mutable external-feed objects on each call."""
    return {
        'AAPL': [
            {'session': '2026-01-05', 'close_usd': '100.00', 'volume': 1000},
            {'session': '2026-01-06', 'close_usd': '102.00', 'volume': 1100},
            {'session': '2026-01-07', 'close_usd': '105.00', 'volume': 1200},
        ],
        'JPM': [
            {'session': '2026-01-05', 'close_usd': '150.00', 'volume': 800},
            {'session': '2026-01-06', 'close_usd': '153.00', 'volume': 900},
            {'session': '2026-01-07', 'close_usd': '156.00', 'volume': 950},
        ],
        'MSFT': [
            {'session': '2026-01-05', 'close_usd': '200.00', 'volume': 1500},
            {'session': '2026-01-06', 'close_usd': '198.00', 'volume': 1600},
            {'session': '2026-01-07', 'close_usd': '202.00', 'volume': 1400},
        ],
    }

print('Immutable market data models initialized.')

Immutable market data models initialized.


### **Analytical Interpretation**

1. **Immutable observation nodes.** `MarketObservation` uses frozen, slotted records containing immutable `str`, `date`, `Decimal`, and `int` values.
2. **Snapshot topology is explicit.** Every instrument maps to an immutable tuple of observations, stored in an outer tuple. No external mutable list is retained by the record format.
3. **Enforced chronology.** Histories must contain unique increasing sessions and terminate at `asof`. No post-decision observation may enter the valuation window in this exercise.
4. **Digest composition.** The digest includes exact serialized decimal representations, time, version, parent digest, and volume. It identifies encoded *content*, not authentic provenance.
5. **Precision contract.** Binary floating-point prices are refused instead of silently converted to `Decimal` with potential representation surprises.
6. **Version immutability.** A corrected observation must create a new object; previous snapshot values cannot be assigned through ordinary APIs.
7. **Security limitation.** Frozen attributes and SHA-256 do not constitute access control or a secure distributed data registry.

In [16]:
# ============================================================
# DEFENSIVE CONSTRUCTION FROM AN EXTERNAL MUTABLE DATA FEED
# ============================================================

from collections.abc import Mapping, Sequence


def build_market_snapshot(
    feed: Mapping[str, Sequence[Mapping[str, object]]],
    *,
    asof: date,
) -> MarketSnapshot:
    """Construct a validated, owned immutable snapshot of feed observations."""
    if not isinstance(feed, Mapping) or not feed:
        raise ValueError('Feed must be a nonempty mapping.')
    groups: list[tuple[str, tuple[MarketObservation, ...]]] = []
    for ticker in sorted(feed):
        rows = feed[ticker]
        if isinstance(rows, (str, bytes)) or not isinstance(rows, Sequence):
            raise TypeError('Every ticker must have a sequence of rows.')
        built = []
        for row in rows:
            if not isinstance(row, Mapping):
                raise TypeError('Every row must be a mapping.')
            required = {'session', 'close_usd', 'volume'}
            if not required <= row.keys():
                raise ValueError('Market row is missing required fields.')
            session = row['session']
            if isinstance(session, str):
                session = date.fromisoformat(session)
            if type(session) is not date:
                raise TypeError('Invalid session type.')
            built.append(MarketObservation(
                ticker=ticker,
                session=session,
                close_usd=require_decimal(row['close_usd']),
                volume=row['volume'],
            ))
        groups.append((ticker, tuple(built)))
    return MarketSnapshot(asof=asof, version=1, observations=tuple(groups))


raw_market_feed = dataset_rows()
asof_29 = date(2026, 1, 7)
snapshot_29_v1 = build_market_snapshot(raw_market_feed, asof=asof_29)
original_digest_29 = snapshot_29_v1.digest

# Input mutation cannot alter a previously constructed snapshot.
raw_market_feed['AAPL'][-1]['close_usd'] = '999.00'
assert dict(snapshot_29_v1.observations)['AAPL'][-1].close_usd == Decimal('105.00')
assert snapshot_29_v1.digest == original_digest_29
print('Initial snapshot version:', snapshot_29_v1.version)
print('As-of date:', snapshot_29_v1.asof)
print('Snapshot isolation confirmed:', snapshot_29_v1.digest == original_digest_29)
print('Initial digest prefix:', snapshot_29_v1.digest[:16])

Initial snapshot version: 1
As-of date: 2026-01-07
Snapshot isolation confirmed: True
Initial digest prefix: 6d85b719addc0425


### **Analytical Interpretation**

1. **Construction breaks aliases.** The builder does not store external row dictionaries or lists; it reads values and creates new immutable observation records.
2. **A snapshot is stable despite feed contamination.** Changing AAPL's current feed price to 999 does not affect the historical snapshot's recorded 105.
3. **As-of constraints are enforced at construction.** The snapshot itself rejects historical series that do not end on the selected session.
4. **Content addressability permits exact version comparisons.** The hexadecimal digest is determined by the canonical serialized values and version metadata, not by unstable `id()` values.
5. **Data timing remains simplified.** A closing price's session date alone cannot prove publication availability; a production point-in-time store must preserve release, vendor receipt, and revision timestamps.

In [17]:
# ============================================================
# CHECKED, COPY-ON-WRITE FINANCIAL DATA CORRECTIONS
# ============================================================

@dataclass(frozen=True, slots=True)
class PriceCorrection:
    correction_id: str
    ticker: str
    session: date
    expected_old_close: Decimal
    revised_close: Decimal
    reason: str

    def __post_init__(self) -> None:
        if any(not isinstance(s, str) or not s.strip() for s in
               (self.correction_id, self.ticker, self.reason)):
            raise ValueError('Correction metadata must be nonempty.')
        if type(self.session) is not date:
            raise TypeError('Correction session must be a date.')
        for item in (self.expected_old_close, self.revised_close):
            if type(item) is not Decimal or not item.is_finite() or item <= 0:
                raise ValueError('Correction prices must be positive finite Decimals.')


def apply_price_corrections(
    base: MarketSnapshot,
    corrections: Sequence[PriceCorrection],
    *,
    expected_parent_digest: str,
) -> MarketSnapshot:
    """Pure checked batch revision; publish nothing on any validation failure.

    Unmodified immutable histories are shared by identity with the parent.
    An all-no-op batch returns the parent unchanged without version bump.
    """
    if expected_parent_digest != base.digest:
        raise ValueError('Stale parent snapshot digest.')
    if not corrections:
        raise ValueError('At least one correction is required.')

    original = dict(base.observations)
    staged = dict(original)
    touched: set[tuple[str, date]] = set()
    correction_ids: set[str] = set()
    modified = False

    for correction in corrections:
        if not isinstance(correction, PriceCorrection):
            raise TypeError('Correction entries must be PriceCorrection objects.')
        if correction.correction_id in correction_ids:
            raise ValueError('Duplicate correction identifier.')
        correction_ids.add(correction.correction_id)
        key = (correction.ticker, correction.session)
        if key in touched:
            raise ValueError('Duplicate ticker/session correction in batch.')
        touched.add(key)
        if correction.ticker not in staged:
            raise ValueError('Unknown ticker.')
        existing = staged[correction.ticker]
        matches = [index for index, obs in enumerate(existing)
                   if obs.session == correction.session]
        if len(matches) != 1:
            raise ValueError('Correction does not match one historical session.')
        index = matches[0]
        old = existing[index]
        if old.close_usd != correction.expected_old_close:
            raise ValueError('Stale expected old close; refusing revision.')
        if old.close_usd == correction.revised_close:
            continue
        candidate = list(existing)
        candidate[index] = replace(old, close_usd=correction.revised_close)
        staged[correction.ticker] = tuple(candidate)
        modified = True

    if not modified:
        return base

    return MarketSnapshot(
        asof=base.asof,
        version=base.version + 1,
        observations=tuple(sorted(staged.items())),
        parent_digest=base.digest,
    )


correction_29 = PriceCorrection(
    correction_id='REV-001',
    ticker='AAPL',
    session=asof_29,
    expected_old_close=Decimal('105.00'),
    revised_close=Decimal('120.00'),
    reason='Synthetic vendor close correction',
)
snapshot_29_v2 = apply_price_corrections(
    snapshot_29_v1, [correction_29], expected_parent_digest=snapshot_29_v1.digest
)
old_histories = dict(snapshot_29_v1.observations)
new_histories = dict(snapshot_29_v2.observations)
assert snapshot_29_v2.version == 2
assert snapshot_29_v2.parent_digest == snapshot_29_v1.digest
assert old_histories['AAPL'] is not new_histories['AAPL']
assert old_histories['JPM'] is new_histories['JPM']
assert old_histories['MSFT'] is new_histories['MSFT']
assert old_histories['AAPL'][-1].close_usd == Decimal('105.00')
assert new_histories['AAPL'][-1].close_usd == Decimal('120.00')
print('Revision committed as new snapshot:', snapshot_29_v2.version)
print('Unchanged immutable JPM history shared:', old_histories['JPM'] is new_histories['JPM'])

Revision committed as new snapshot: 2
Unchanged immutable JPM history shared: True


### **Analytical Interpretation**

1. **Pure publication:** a correction creates a new `MarketSnapshot`; the parent remains untouched.
2. **Optimistic version checking:** `expected_parent_digest` rejects a caller that is revising an unexpected state.
3. **Per-observation preconditions:** `expected_old_close` detects stale data or wrong correction targets rather than silently overwriting prices.
4. **All-or-nothing validation:** local staging updates are discarded if a later correction raises an exception; the base object never changes.
5. **Intentional structural sharing:** unchanged JPM/MSFT history tuples retain identity because every observation and the tuple itself is immutable.
6. **No-op behavior:** a correction replacing a value with an equal value returns the original snapshot, without manufacturing a new revision.
7. **Caveat:** the function is pure and thus safe against ordinary in-process alias mutation of existing snapshots. An external database writer still needs transactional durability and coordination.

In [18]:
# ============================================================
# ASSET VALUATION, BASKET RETURNS, AND RECONCILIATION
# ============================================================

@dataclass(frozen=True, slots=True)
class Holding:
    ticker: str
    shares: int

    def __post_init__(self) -> None:
        if not isinstance(self.ticker, str) or not self.ticker.strip():
            raise ValueError('Holding ticker must be nonempty.')
        if type(self.shares) is not int or self.shares < 0:
            raise ValueError('Long-only shares must be nonnegative integers.')


def portfolio_nav(
    snapshot: MarketSnapshot,
    holdings: tuple[Holding, ...],
    *,
    cash_usd: Decimal,
) -> Decimal:
    """Mark long-only holdings to the snapshot as-of close."""
    if type(cash_usd) is not Decimal or not cash_usd.is_finite():
        raise ValueError('Cash must be a finite Decimal.')
    if not isinstance(holdings, tuple) or any(type(h) is not Holding for h in holdings):
        raise TypeError('Holdings must be a tuple of Holding records.')
    names = [h.ticker for h in holdings]
    if len(names) != len(set(names)):
        raise ValueError('Duplicate holding tickers.')
    histories = dict(snapshot.observations)
    if not set(names) <= histories.keys():
        raise ValueError('Holding ticker missing from market snapshot.')
    return cash_usd + sum(
        (histories[h.ticker][-1].close_usd * h.shares for h in holdings),
        Decimal('0'),
    )


def equal_weight_interval_return(snapshot: MarketSnapshot) -> Decimal:
    """Arithmetic mean of cumulative asset price returns over each history."""
    returns = [series[-1].close_usd / series[0].close_usd - Decimal(1)
               for _, series in snapshot.observations]
    return sum(returns, Decimal(0)) / Decimal(len(returns))


holdings_29 = (
    Holding('AAPL', 10),
    Holding('JPM', 8),
    Holding('MSFT', 5),
)
cash_29 = Decimal('1000.00')
nav_before_29 = portfolio_nav(snapshot_29_v1, holdings_29, cash_usd=cash_29)
nav_after_29 = portfolio_nav(snapshot_29_v2, holdings_29, cash_usd=cash_29)
return_before_29 = equal_weight_interval_return(snapshot_29_v1)
return_after_29 = equal_weight_interval_return(snapshot_29_v2)

assert nav_before_29 == Decimal('4308.00')
assert nav_after_29 == Decimal('4458.00')
assert nav_after_29 - nav_before_29 == Decimal('150.00')
assert return_before_29 == Decimal('0.03333333333333333333333333333')
assert return_after_29 == Decimal('0.08333333333333333333333333333')

print('BASELINE NAV:', nav_before_29)
print('REVISED NAV:', nav_after_29)
print('MARK-TO-MARKET REVISION IMPACT:', nav_after_29 - nav_before_29)
print('BASELINE EQUAL-WEIGHT RETURN:', f'{return_before_29:.4%}')
print('REVISED EQUAL-WEIGHT RETURN:', f'{return_after_29:.4%}')

BASELINE NAV: 4308.00
REVISED NAV: 4458.00
MARK-TO-MARKET REVISION IMPACT: 150.00
BASELINE EQUAL-WEIGHT RETURN: 3.3333%
REVISED EQUAL-WEIGHT RETURN: 8.3333%


### **Analytical Interpretation**

1. **Pure valuation:** the NAV function reads snapshot values but modifies neither holdings nor market observations.
2. **Financial identity:** `NAV = 1000 cash + 10×105 AAPL + 8×156 JPM + 5×202 MSFT = $4,308`.
3. **One checked price change:** increasing AAPL's close by $15 for 10 shares increases NAV by **$150**, from $4,308 to $4,458.
4. **Synthetic equal-weight interval return:** original `(5% + 4% + 1%)/3 = 3.3333%`; revised `(20% + 4% + 1%)/3 = 8.3333%`.
5. **Interpretation:** the return difference is caused by a data revision, not an executed trading strategy or predictive signal.
6. **No drift from source mutation:** the original version's valuation still uses AAPL $105, despite the external feed having been changed earlier.
7. **Precision:** `Decimal` eliminates representation artifacts of binary floating point at the input boundary but does not eliminate all possible rounding; division still depends on the active decimal context.

In [19]:
# ============================================================
# IDENTITY-BASED SHARED MUTABLE OBJECT GRAPH AUDITOR
# ============================================================

from collections.abc import Mapping


def shared_mutable_paths(roots: Mapping[str, object]) -> tuple[tuple[str, ...], ...]:
    """Report paths to repeated dict/list/set objects in supplied roots.

    Scope: built-in dict/list/set/tuple traversal only; not a complete
    heap profiler. Recursion stacks detect cycles. ID values are not
    returned and objects remain referenced by the roots during scanning.
    """
    groups: dict[int, list[str]] = {}
    def walk(value: object, path: str, active: set[int]) -> None:
        if isinstance(value, (dict, list, set)):
            groups.setdefault(id(value), []).append(path)
        marker = id(value)
        if marker in active:
            return
        if isinstance(value, (dict, list, tuple)):
            active.add(marker)
            try:
                if isinstance(value, dict):
                    for key, member in value.items():
                        walk(member, f'{path}[{key!r}]', active)
                else:
                    for index, member in enumerate(value):
                        walk(member, f'{path}[{index}]', active)
            finally:
                active.remove(marker)
    for name, root in roots.items():
        walk(root, name, set())
    return tuple(sorted(
        (tuple(sorted(paths)) for paths in groups.values() if len(paths) > 1),
        key=lambda paths: paths[0],
    ))


shared_feed = dataset_rows()
second_view = {'AAPL': shared_feed['AAPL']}
findings_29 = shared_mutable_paths({
    'vendor_feed': shared_feed,
    'research_view': second_view,
})
assert findings_29
print('SHARED MUTABLE REFERENCES FOUND:', len(findings_29))
for group in findings_29[:5]:
    print('  alias paths:', '  <=>  '.join(group))

# Converting to immutable MarketSnapshot records avoids sharing any
# externally mutable price list or row dictionary from this source.
safe_29 = build_market_snapshot(shared_feed, asof=asof_29)
shared_feed['AAPL'][0]['close_usd'] = '777.00'
assert dict(safe_29.observations)['AAPL'][0].close_usd == Decimal('100.00')
print('Snapshot isolated from mutable vendor feed:', True)

SHARED MUTABLE REFERENCES FOUND: 4
  alias paths: research_view['AAPL']  <=>  vendor_feed['AAPL']
  alias paths: research_view['AAPL'][0]  <=>  vendor_feed['AAPL'][0]
  alias paths: research_view['AAPL'][1]  <=>  vendor_feed['AAPL'][1]
  alias paths: research_view['AAPL'][2]  <=>  vendor_feed['AAPL'][2]
Snapshot isolated from mutable vendor feed: True


### **Analytical Interpretation**

1. **Shared identity is a graph property.** The auditor records multiple paths reaching one mutable object; it does not compare contents.
2. **Live object lifetimes protect diagnostic IDs.** The roots maintain strong references while the graph is traversed; IDs are used internally and never exported as persistent identifiers.
3. **Cycle safety.** A per-path active set prevents infinite traversal of lists/dictionaries containing themselves.
4. **Conservative scope.** The scanner handles selected built-in containers only. It does not inspect custom objects, native buffers, weak references, object attributes, or aliasing through memory views.
5. **Snapshot construction provides a stronger contract.** Immutable observation records eliminate the external alias paths relevant to price values, not just the outer container alias.
6. **Operational caution:** automatically traversing arbitrary user-defined objects can invoke methods and side effects; production graph inspection needs a deliberately limited object policy.

## 2.9.15.1 Expected Results and Financial Reconciliation

| Measure | Baseline snapshot (v1) | Revised snapshot (v2) |
|---|---:|---:|
| AAPL last close | $105.00 | $120.00 |
| JPM last close | $156.00 | $156.00 |
| MSFT last close | $202.00 | $202.00 |
| AAPL / JPM / MSFT holdings | 10 / 8 / 5 | Unchanged |
| Cash | $1,000.00 | $1,000.00 |
| Portfolio NAV | **$4,308.00** | **$4,458.00** |
| Equal-weight interval price return | **3.3333%** | **8.3333%** |
| Revision mark-to-market impact | — | **+$150.00** |
| Snapshot version | 1 | 2 |
| Parent digest | None | v1 digest |

### Required Reconciliation

\[
\Delta \mathrm{NAV} = q_{\mathrm{AAPL}} (p_{\mathrm{AAPL},v2} - p_{\mathrm{AAPL},v1})
= 10(120-105)=150\ \mathrm{USD}
\]

**Do not interpret $150 as trading profit.** It is a revision-induced change in mark-to-market valuation of fixed synthetic holdings. No order was placed. Under a real point-in-time research protocol, the revised historical number must not silently replace the information that was available to a researcher at the original decision time.

### Complexity and Ownership

- Snapshot construction: **O(N)** observation reconstruction, plus sorting instruments **O(A log A)** for A tickers and N total observations.
- Revision: copies the top-level instrument mapping **O(A)** and each changed ticker's immutable history **O(T_i)**; unchanged history tuples are reused.
- Valuation: **O(A + H)** to build a temporary lookup and value H holdings.
- Memory: snapshot versions share unchanged immutable observation histories; deep-copying the entire input feed is unnecessary.
- SHA-256 digest: **O(N)** over canonical serialized observation data.
- Thread safety: published immutable snapshots can be safely read under the stated record contract; a shared external mutable feed still requires an explicit ingestion/coordination policy.

## 2.9.16 Automated Testing and Financial Invariants

### Test Design

**Identity/alias invariants** — distinct outer objects may share mutable nested objects; deep copy preserves within-clone alias topology and cycles; copy-on-write versions share only unchanged immutable histories.

**Snapshot invariants** — previous versions remain unchanged after source-feed changes, corrections, and rejected batches; digest/version linkage is internally consistent.

**Correction invariants** — stale parent digest, stale expected price, duplicate correction IDs/targets, missing tickers/sessions, invalid prices, and invalid input records cause failures with no changes to the base snapshot.

**Financial invariants** — NAV equals cash plus the sum of price×shares; correction impact equals shares×price delta; equal-weight interval returns match independently derived values; duplicate positions and missing market data fail.

**Scope:** These tests establish selected deterministic invariants, not data-vendor authenticity, zero security vulnerabilities, database durability, or correctness under arbitrary adversarial execution.

In [20]:
# ============================================================
# INDEPENDENT REGRESSION AND FAILURE-PATH TEST SUITE
# ============================================================

import copy
import unittest


class ObjectIdentityAndMarketRevisionTests(unittest.TestCase):
    def setUp(self):
        self.feed = dataset_rows()
        self.asof = date(2026, 1, 7)
        self.base = build_market_snapshot(self.feed, asof=self.asof)
        self.correction = PriceCorrection(
            'REV-001', 'AAPL', self.asof,
            Decimal('105.00'), Decimal('120.00'), 'vendor correction',
        )
        self.holdings = (Holding('AAPL', 10), Holding('JPM', 8), Holding('MSFT', 5))

    def revise(self, *corrections):
        return apply_price_corrections(
            self.base, corrections, expected_parent_digest=self.base.digest
        )

    def test_identity_vs_equality(self):
        x, y = [1, 2], [1, 2]
        self.assertEqual(x, y)
        self.assertIsNot(x, y)

    def test_alias_mutation(self):
        x = [1]
        y = x
        y.append(2)
        self.assertIs(x, y)
        self.assertEqual(x, [1, 2])

    def test_shallow_copy_retains_nested_alias(self):
        original = {'a': [1]}
        copied = copy.copy(original)
        self.assertIsNot(copied, original)
        self.assertIs(copied['a'], original['a'])

    def test_deepcopy_preserves_internal_alias(self):
        nested = [1]
        original = [nested, nested]
        copied = copy.deepcopy(original)
        self.assertIs(copied[0], copied[1])
        self.assertIsNot(copied[0], nested)

    def test_deepcopy_preserves_cycle(self):
        original = []
        original.append(original)
        copied = copy.deepcopy(original)
        self.assertIs(copied[0], copied)
        self.assertIsNot(copied, original)

    def test_immutable_tuple_can_contain_mutable_list(self):
        outer = ([1],)
        outer[0].append(2)
        self.assertEqual(outer[0], [1, 2])

    def test_stable_hashable_key(self):
        key = InstrumentKey('AAPL', 'NASDAQ')
        self.assertEqual({key: 10}[InstrumentKey('AAPL', 'NASDAQ')], 10)

    def test_weak_ref_owning_reference(self):
        obj = MarketCacheEntry('unit-test')
        weak = weakref.ref(obj)
        alias = obj
        del obj
        self.assertIs(weak(), alias)

    def test_live_proxy_reflects_original_mutation(self):
        mutable = {'x': [1]}
        proxy = MappingProxyType(mutable)
        mutable['y'] = [2]
        self.assertIn('y', proxy)

    def test_frozen_dataclass_shallow(self):
        record = ShallowFrozenState({'AAPL': [1]})
        record.prices['AAPL'].append(2)
        self.assertEqual(record.prices['AAPL'], [1, 2])

    def test_source_feed_isolation(self):
        self.feed['AAPL'][-1]['close_usd'] = '999.00'
        self.assertEqual(dict(self.base.observations)['AAPL'][-1].close_usd, Decimal('105.00'))

    def test_source_row_isolation(self):
        self.feed['JPM'][0]['volume'] = 999999
        self.assertEqual(dict(self.base.observations)['JPM'][0].volume, 800)

    def test_independent_input_equal_digest(self):
        other = build_market_snapshot(dataset_rows(), asof=self.asof)
        self.assertEqual(other.digest, self.base.digest)

    def test_fingerprint_is_hex(self):
        self.assertEqual(len(self.base.digest), 64)
        self.assertTrue(all(char in '0123456789abcdef' for char in self.base.digest))

    def test_version_and_parent_digest(self):
        updated = self.revise(self.correction)
        self.assertEqual(updated.version, 2)
        self.assertEqual(updated.parent_digest, self.base.digest)
        self.assertNotEqual(updated.digest, self.base.digest)

    def test_revised_history_is_new(self):
        updated = self.revise(self.correction)
        self.assertIsNot(dict(updated.observations)['AAPL'], dict(self.base.observations)['AAPL'])

    def test_unchanged_histories_shared(self):
        updated = self.revise(self.correction)
        for ticker in ('JPM', 'MSFT'):
            self.assertIs(dict(updated.observations)[ticker], dict(self.base.observations)[ticker])

    def test_old_snapshot_preserved(self):
        updated = self.revise(self.correction)
        self.assertEqual(dict(self.base.observations)['AAPL'][-1].close_usd, Decimal('105.00'))
        self.assertEqual(dict(updated.observations)['AAPL'][-1].close_usd, Decimal('120.00'))

    def test_noop_returns_same_object(self):
        noop = PriceCorrection('REV-NOOP', 'AAPL', self.asof,
                               Decimal('105.00'), Decimal('105.00'), 'no change')
        self.assertIs(self.revise(noop), self.base)

    def test_wrong_parent_digest_rejected(self):
        with self.assertRaises(ValueError):
            apply_price_corrections(self.base, [self.correction], expected_parent_digest='0' * 64)

    def test_stale_expected_price_rejected(self):
        wrong = replace(self.correction, expected_old_close=Decimal('106.00'))
        with self.assertRaises(ValueError):
            self.revise(wrong)

    def test_unknown_ticker_rejected(self):
        unknown = replace(self.correction, ticker='NOPE')
        with self.assertRaises(ValueError):
            self.revise(unknown)

    def test_unknown_session_rejected(self):
        unknown = replace(self.correction, session=date(2025, 1, 1))
        with self.assertRaises(ValueError):
            self.revise(unknown)

    def test_duplicate_correction_identifier_rejected(self):
        with self.assertRaises(ValueError):
            self.revise(self.correction, self.correction)

    def test_duplicate_ticker_session_rejected(self):
        duplicate = replace(self.correction, correction_id='REV-002')
        with self.assertRaises(ValueError):
            self.revise(self.correction, duplicate)

    def test_failed_late_batch_leaves_parent_unchanged(self):
        base_digest = self.base.digest
        stale_later = PriceCorrection('REV-003', 'JPM', self.asof,
                                      Decimal('999'), Decimal('160'), 'stale')
        with self.assertRaises(ValueError):
            self.revise(self.correction, stale_later)
        self.assertEqual(self.base.digest, base_digest)
        self.assertEqual(dict(self.base.observations)['AAPL'][-1].close_usd, Decimal('105.00'))

    def test_empty_revision_batch_rejected(self):
        with self.assertRaises(ValueError):
            self.revise()

    def test_float_price_is_rejected(self):
        self.feed['AAPL'][0]['close_usd'] = 100.0
        with self.assertRaises(TypeError):
            build_market_snapshot(self.feed, asof=self.asof)

    def test_boolean_volume_is_rejected(self):
        self.feed['AAPL'][0]['volume'] = True
        with self.assertRaises(ValueError):
            build_market_snapshot(self.feed, asof=self.asof)

    def test_unordered_sessions_rejected(self):
        self.feed['AAPL'][0], self.feed['AAPL'][1] = self.feed['AAPL'][1], self.feed['AAPL'][0]
        with self.assertRaises(ValueError):
            build_market_snapshot(self.feed, asof=self.asof)

    def test_duplicate_sessions_rejected(self):
        self.feed['AAPL'][1]['session'] = self.feed['AAPL'][0]['session']
        with self.assertRaises(ValueError):
            build_market_snapshot(self.feed, asof=self.asof)

    def test_missing_asof_rejected(self):
        self.feed['AAPL'][-1]['session'] = '2026-01-06'
        with self.assertRaises(ValueError):
            build_market_snapshot(self.feed, asof=self.asof)

    def test_empty_source_rejected(self):
        with self.assertRaises(ValueError):
            build_market_snapshot({}, asof=self.asof)

    def test_invalid_correction_price_rejected(self):
        with self.assertRaises(ValueError):
            PriceCorrection('R', 'AAPL', self.asof, Decimal('105'), Decimal('NaN'), 'invalid')

    def test_initial_nav(self):
        self.assertEqual(portfolio_nav(self.base, self.holdings, cash_usd=Decimal('1000')),
                         Decimal('4308.00'))

    def test_revised_nav(self):
        updated = self.revise(self.correction)
        self.assertEqual(portfolio_nav(updated, self.holdings, cash_usd=Decimal('1000')),
                         Decimal('4458.00'))

    def test_nav_impact_reconciles(self):
        updated = self.revise(self.correction)
        impact = (portfolio_nav(updated, self.holdings, cash_usd=Decimal('1000'))
                  - portfolio_nav(self.base, self.holdings, cash_usd=Decimal('1000')))
        self.assertEqual(impact, Decimal(10) * Decimal(15))

    def test_equal_weight_returns(self):
        updated = self.revise(self.correction)
        self.assertEqual(equal_weight_interval_return(self.base), Decimal(1) / Decimal(30))
        self.assertEqual(equal_weight_interval_return(updated), Decimal(1) / Decimal(12))

    def test_duplicate_holdings_rejected(self):
        with self.assertRaises(ValueError):
            portfolio_nav(self.base, (Holding('AAPL', 1), Holding('AAPL', 2)), cash_usd=Decimal('0'))

    def test_missing_market_price_rejected(self):
        with self.assertRaises(ValueError):
            portfolio_nav(self.base, (Holding('NOPE', 5),), cash_usd=Decimal('0'))

    def test_bool_shares_rejected(self):
        with self.assertRaises(ValueError):
            Holding('AAPL', True)

    def test_alias_auditor_detects_shared_list(self):
        underlying = [1, 2]
        result = shared_mutable_paths({'a': {'x': underlying}, 'b': {'y': underlying}})
        self.assertTrue(any(len(paths) >= 2 for paths in result))

    def test_alias_auditor_handles_cycles(self):
        cycle = []
        cycle.append(cycle)
        result = shared_mutable_paths({'root': cycle})
        self.assertTrue(result)

    def test_multi_ticker_revision(self):
        other = PriceCorrection('REV-JPM', 'JPM', self.asof,
                                Decimal('156.00'), Decimal('160.00'), 'update')
        updated = self.revise(self.correction, other)
        self.assertEqual(dict(updated.observations)['JPM'][-1].close_usd, Decimal('160.00'))
        self.assertIs(dict(updated.observations)['MSFT'], dict(self.base.observations)['MSFT'])

    def test_immutability_of_observation(self):
        from dataclasses import FrozenInstanceError
        obs = dict(self.base.observations)['AAPL'][-1]
        with self.assertRaises(FrozenInstanceError):
            obs.close_usd = Decimal('1000')

    def test_snapshot_tuples(self):
        self.assertIsInstance(self.base.observations, tuple)
        self.assertTrue(all(isinstance(series, tuple) for _, series in self.base.observations))

    def test_contiguous_revision_chain(self):
        second = self.revise(self.correction)
        third_corr = PriceCorrection('REV-SECOND', 'AAPL', self.asof,
                                     Decimal('120'), Decimal('121'), 'second revision')
        third = apply_price_corrections(second, [third_corr], expected_parent_digest=second.digest)
        self.assertEqual(third.version, 3)
        self.assertEqual(third.parent_digest, second.digest)
        self.assertEqual(dict(self.base.observations)['AAPL'][-1].close_usd, Decimal('105.00'))


suite_29 = unittest.defaultTestLoader.loadTestsFromTestCase(ObjectIdentityAndMarketRevisionTests)
result_29 = unittest.TextTestRunner(verbosity=1).run(suite_29)
assert result_29.wasSuccessful()
print('Verified automated tests:', result_29.testsRun)

.

.

.

.

.

.

... (39 outputs left out)

.

.

----------------------------------------------------------------------
Ran 47 tests in 0.023s

OK


Verified automated tests: 47


### **Analytical Interpretation**

1. **Reference correctness:** tests distinguish identity, equality, shared nested aliases, reconstructed cycles, and intentionally shared immutable histories.
2. **Source isolation:** mutating original price/volume dictionaries after construction does not change prior snapshots.
3. **Optimistic revisions:** stale parent digests and stale observed prices are rejected.
4. **Atomic batch behavior of the *pure transformation*:** if a later correction fails, a modified published snapshot never exists; the input state remains unchanged.
5. **No-op revisions:** equal-value corrections do not increment version numbers or create artificial new identities.
6. **Financial invariants:** baseline NAV **$4,308.00**, revised NAV **$4,458.00**, and delta **$150.00** reconcile to shares×price difference.
7. **Input contracts:** booleans as integer volumes/shares, binary-float prices, duplicate dates, missing data, and invalid decimals fail explicitly.
8. **Scope:** the tests do not prove security isolation, vendor accuracy, or crash-safe publication; those require persistence-layer controls, signatures, and external audit governance.

## 2.9.17 Final Analytical Interpretation

### **Analytical Interpretation**

**1. Identity vs. equality.** A reference relationship (`is`) and value semantics (`==`) answer different questions. A correct quantitative dataset may contain equal but independent observations, or distinct containers with shared nested references.

**2. Aliasing is an object-graph problem.** Single-variable reasoning is insufficient once a dataset is reachable through multiple mappings, list elements, closures, and callbacks. Graph topology determines which mutations propagate.

**3. Mutability is orthogonal to binding.** Reassigning a name changes a reference; mutating a container changes state visible through aliases. A frozen container can still reference mutable contents.

**4. Hashability is a contract.** Equal hashable keys must have equal hashes and stable hash values during use. Never use a mutable market price as part of a key's value-based hash.

**5. Shallow copy is not isolation.** A new outer mapping can still share price arrays with the source. A read-only mapping proxy may also reflect upstream modifications.

**6. Deep copy preserves topology.** The memo mechanism handles cycles and repeated references. It may nevertheless duplicate too much and cannot by itself define business ownership of caches, sockets, or external data resources.

**7. Custom copy policies require domain awareness.** Immutable calibration data may safely be shared, while mutable quote buffers may need cloning. Copy policy must be explicit and tested.

**8. Failed operations may mutate first.** Augmented assignment can alter a nested mutable object before an immutable outer target refuses assignment-back. Exceptions are not automatic rollback.

**9. Function calls are call-by-sharing.** Mutation of arguments affects the caller's object; local parameter rebinding does not change the caller's binding. Mutable defaults silently preserve state across invocations.

**10. Weak references describe reachability, not ownership guarantees.** Weak caches are useful for optional reuse but unsuitable as the sole storage for authoritative financial data.

**11. Memory diagnostics are measurements.** Refcounts, traced allocations, and finalization timing depend on runtime and execution context; semantic identity tests are more portable than numeric ID assumptions.

**12. Concurrent access requires coordination.** The GIL does not turn a logically multi-step financial update into an atomic transaction. Critical sections or immutable published versions make ownership clearer.

**13. Copy-on-write preserves prior research versions.** Under immutable-record contracts, unchanged histories can be shared while revised observations are independently constructed.

**14. Content digests aid reproducibility.** A canonical digest makes version identity checkable; it does not certify that a data vendor or exchange supplied correct observations.

**15. Point-in-time research requires availability metadata.** Session-date validity is only a first step; true prevention of look-ahead bias requires release/receipt/revision timestamps and sound model evaluation design.

**16. Financial interpretation matters.** A synthetic $150 NAV change caused by a corrected quote is not realized P&L, strategy alpha, or execution profit.

**17. Engineering objective.** Design systems where data ownership, aliasing, immutability boundaries, revision lineage, numerical contracts, and exceptional behavior can all be inspected and verified independently.

### Final Assessment

Expert-level Python requires reasoning about **the identity and reachability of objects across an entire program**, not merely the spelling of variables. For quantitative research, controlled immutable snapshots and explicit revision contracts are foundational to reproducibility, auditability, and correct financial interpretation.

## Advanced Practice Exercises — Independent Work

### Exercise 1 — Identity-Aware Object Graph Profiler

Build a bounded traversal of user-selected `dict`, `list`, `tuple`, and dataclass records. Record multiple paths to each **live mutable node**, detect cycles, and report shared references without exporting `id()` as a durable identifier. Include separate tests for value-equal independent objects, self-cycles, and repeated aliases. **Challenge:** distinguish alias detection from traversal of intentionally shared immutable nodes.

### Exercise 2 — Deep Copy Fidelity and Custom Protocols

Implement a class with owned mutable data and shared immutable policy records. Provide correct `__copy__`/`__deepcopy__` behavior, preserve internal repeated references and cycles with `memo`, and prove that cloned mutations cannot affect the source. Explain why deep-copying external connection objects or callbacks may be inappropriate.

### Exercise 3 — Python Hashing and Mutable-Key Failure Modes

Construct a deliberately unsafe value-based mutable key in an isolated test and demonstrate lookup instability after mutation. Refactor the design using an immutable instrument identifier. Verify `x == y ⇒ hash(x) == hash(y)` and document why relying on `unsafe_hash=True` is risky. Never use the broken key in a production collection.

### Exercise 4 — Shallow Freeze vs. Transitive Immutability

Compare a frozen dataclass containing a list, a tuple containing a dictionary, an externally backed `MappingProxyType`, and an independently constructed immutable record graph. Attempt mutations through all known external aliases. Produce a table of which designs preserve **historical snapshot integrity** and which merely prevent one kind of assignment.

### Exercise 5 — Copy-on-Write Historical Price Revisions

Extend `PriceCorrection` to revise **volume and close**, enforce strict expected-old-field preconditions, and reject duplicate ticker/session revisions even when the fields differ. Measure which immutable histories can be structurally shared, and test that failed changes preserve the original digest and NAV.

### Exercise 6 — Revision Lineage and Fork Detection

Build a revision-graph validator that checks parent digests, version increments, and repeated data revisions. Support deliberate branching from one source version. Detect attempts to merge conflicting corrections for the same ticker/session; do not overwrite a conflicting history silently.

### Exercise 7 — Content Addressing and Canonical Encoding

Define a canonical serialization format with a schema identifier, exact decimal encoding policy, date format, and deterministic instrument ordering. Verify fingerprints across independent reconstructions. Explain why a hash does not establish authenticity and propose signed metadata/provenance controls.

### Exercise 8 — Point-in-Time Data Availability

Extend observations with `vendor_received_at`, `exchange_published_at`, and `revision_known_at`. Implement a query that returns precisely the data version known at a specified historical decision timestamp. Test delayed data publication, corrected prices, market holidays, and out-of-order vendor arrival. Do not conflate observation date with availability time.

### Exercise 9 — Deterministic Concurrency and Lost Updates

Force a two-thread lost update using barriers as in §2.9.14. Repair it with a lock and then redesign the update to publish a new immutable state snapshot. Compare the guarantees of a mutex, an optimistic version check, and a database transaction, including process crash and distributed-writer limits.

### Exercise 10 — Weak Cache Ownership Audit

Build a `WeakValueDictionary` cache for reproducible, disposable analysis results. Demonstrate that the cache entry can disappear when no strong owner exists. Separate authoritative serialized snapshots from optional cache entries. Verify that no economic calculation depends on garbage-collection timing.

### Exercise 11 — Memory and Structural-Sharing Benchmark

Compare the time and traced Python allocation cost of `deepcopy` versus copy-on-write for 10,000 synthetic immutable observations with a 1% revision rate. Use `tracemalloc` and `time.perf_counter` in repeated trials, report median and variability, and acknowledge that traced Python memory is not full process RSS.

### Exercise 12 — End-to-End Quant Research Reproducibility

Create a synthetic momentum strategy that consumes immutable point-in-time snapshots. Run research on v1 and v2, explain exactly which results differ after a historical correction, and assert that v1 results remain unchanged. Separate signal timestamps, evaluation timestamps, revision availability, holdings, and execution assumptions. Include independent numerical regression tests.

## Expert Challenge — Versioned Quantitative Research Data Infrastructure

### Objective

Design a reusable **immutable market-data repository with explicit alias-ownership, correction lineage, financial valuation, and reproducibility guarantees**. Unlike a simple deep-copy utility, the repository must answer: *Which data were available at decision time? Which version was used for each research result? Which mutable references can still alter the output?*

### Deliverables

1. Immutable `PriceObservation`, `DataVersion`, and `PortfolioPosition` domain models.
2. Strict decimal and timestamp validation; rejection of silent binary-float conversions.
3. Deterministic canonical content addressing with explicit schema/version metadata.
4. An immutable historical version DAG with parent digests and verified lineage.
5. Checked copy-on-write correction batches with optimistic concurrency semantics.
6. Structural-sharing analysis: document when reference sharing is safe.
7. A bounded, safe object-graph auditor for selected owned mutable containers.
8. Point-in-time availability reconstruction accounting for vendor revisions.
9. Price/volume revision audit trail and provenance metadata.
10. Pure NAV, exposure, and equal-weight basket computations.
11. Invariant-driven reconciliation and revision impact attribution.
12. Unit, property-based (if optionally installed), and adversarial failure-path tests.
13. Concurrency tests and an explicit publication/synchronization contract.
14. Repeated memory/time profiling with environment documentation.
15. GitHub-ready package layout, user documentation, and CI workflow.

### Required Invariants

- No published snapshot changes through ordinary application operations.
- Every published version has a reproducible digest.
- Every revision records a validated parent and expected old values.
- A rejected revision leaves all prior versions and results unchanged.
- Unmodified immutable observations can be shared without risk of source mutation.
- NAV differences reconcile to documented market-data and holdings changes.
- Decision-time research never substitutes revisions unavailable at that decision time.
- A cache miss or garbage-collection event never changes the correct research result.

### Evaluation Rubric

| Dimension | Weight |
|---|---:|
| Object identity, aliasing and mutability correctness | 20% |
| Immutable versioning and correction lineage | 20% |
| Temporal integrity and reproducibility | 20% |
| Financial calculations and reconciliation | 15% |
| Failure/concurrency testing | 15% |
| Software architecture and documentation | 10% |

### Final Goal

Deliver a research system whose **state ownership is explicit**, whose historical data versions remain reproducible, and whose financial results can be independently re-derived. Treat identity, hashing, copying, concurrency, and point-in-time data availability as separate engineering concerns rather than interchangeable safeguards.